# ANALYSE DU STOCK ET DES VENTES DU SITE BOTTLENECK

# OBJECTIF DE CE NOTEBOOK

Bienvenue dans l'outil plébiscité par les analystes de données Jupyter.

Il s'agit d'un outil permettant de mixer et d'alterner code, texte et graphiques.

Cet outil est formidable pour plusieurs raisons:

+ Il permet de tester des lignes de codes au fur et à mesure de votre rédaction, de constater immédiatement le résultat d'une instruction, de la corriger si nécessaire.
+ Il permet aussi de rédiger du texte pour expliquer l'approche suivie ou les résultats d'une analyse et de le mettre en forme grâce à du code html ou plus simple avec **Markdown**
+ Il est possible d'ajouter des graphiques

Pour vous aider dans vos premiers pas à l'usage de Jupyter et de Python, nous avons rédigé ce notebook en vous indiquant les instructions à suivre.

Il vous suffit pour cela de saisir le code Python répondant à l'instruction donnée.

Vous verrez de temps à autre le code Python répondant à une instruction donnée mais cela est fait pour vous aider à comprendre la nature du travail qui vous est demandé.

Et gardez à l'esprit qu'il n'y a pas de solution unique pour résoudre un problème et qu'il y a autant de résolutions de problèmes que de développeurs ;)...



# Etape 1 - Importation des librairies et chargement des fichiers

## Importation des librairies

In [1]:
#Importation de la librairie Pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Importation de la librairie plotly express
import plotly.express as px  # permet de créer des graphiques interactifs avec peu de code.


In [ ]:
#permet d'aller prendre les fichiers directement dans mon drive
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/My Drive/Colab Notebooks/

In [ ]:
#Trouver dans Google l'instruction permettant d'afficher toutes les colonnes d'un dataframe
#Saisir dans Google les mots clés "display all columns dataframe Pandas" par exemple.
#Dans les résultats de la recherche, privilégier les solutions provenant de Stack Overflow ou Medium
pd.set_option('display.max_columns', None) #dit à pandas d’afficher toutes les colonnes, none supprime la limite

## Chargements des fichiers

In [ ]:
#Importation du fichier web.xlsx
df_web = pd.read_excel("web.xlsx")
#Importation du fichier erp.xlsx
df_erp = pd.read_excel("erp.xlsx")
#Importation du fichier liaison.xlsx
df_liaison = pd.read_excel("liaison.xlsx")

# Etape 2 - Analyse exploratoire des fichiers

## 2.1 - Analyse exploratoire du fichier erp.xlsx

In [ ]:
#Afficher les dimensions du dataset
print("Le tableau comporte {} observation(s) ou article(s)".format(df_erp.shape[0]))
print("Le tableau comporte {} colonne(s)".format(df_erp.shape[1]))

In [ ]:
#Consulter le nombre de colonnes
print("Le nombre de colonnes est de : ",df_erp.shape[1],"\n")
#La nature des données dans chacune des colonnes
print("Nature des données pour chaques colonnes :\n", df_erp.dtypes,"\n")
#Le nombre de valeurs présentes dans chacune des colonnes
print("Nombre de valeurs par colonnes :\n", df_erp.count())


In [ ]:
#Afficher les 5 premières lignes de la table
df_erp.head()


In [ ]:
#Vérifier si il y a des lignes en doublon dans la colonne product_id
print("il y a : ", df_erp.duplicated(subset=["product_id"]).sum(), " doublons")
#.duplicated() va renvoyer pour chaque ligne de product_id True si dupliqué False sinon sachant que True = 1 et False = 0
# on fait ensuite la somme donc si 0 pas de doublons



In [ ]:
#Afficher les valeurs distinctes de la colonne stock_status
df_erp["stock_status"].unique()
#À quelle(s) autre(s) colonne(s) sont-elles liées ?
#Elle est liée à la colonne stock_quantity

In [ ]:
#Création d'une colonne "stock_status_2"
#La valeur de cette deuxième colonne sera fonction de la valeur dans la colonne "stock_quantity"
#Si la valeur de la colonne "stock_quantity" est nulle, renseigner "outofstock" sinon mettre "instock"
df_erp["stock_status_2"] = np.where(df_erp["stock_quantity"] <= 0, "outofstock", "instock") #permet de créer une colonne ou "outofstock" est retourné si condition est vrai sinon "instock"
df_erp

In [ ]:
#Vérifions que les 2 colonnes sont identiques:
#Les 2 colonnes sont strictement identiques si les valeurs de chaque ligne sont strictement identiques 2 à 2
#La comparaison de 2 colonnes peut se réaliser simplement avec l'instruction ci-dessous:
df_erp["stock_status"] == df_erp["stock_status_2"]

#Le résultat est l'affichage de True ou False pour chacune des lignes du dataset
#C'est un bon début, mais difficile à exploiter

In [ ]:
#Mais il est possible de synthétiser ce résultat en effectuant la somme de cette colonne:
#True vaut 1 et False 0
#Nous devrions obtenir la somme de 825 qui correspond au nombre de lignes dans ce dataset
somme_stock_status = df_erp["stock_status"] == df_erp["stock_status_2"]
print("Nombre de lignes identiques : ", somme_stock_status.sum(), "sur 825")

#afficher la donnée erronée
print("\n", "Voici les données erronées :")
df_erp[somme_stock_status == False]

In [ ]:
#Si les colonnes ne sont absolument pas identiques ligne à ligne alors identifier la ligne en écart
##Dans ce cas je vous donne ce lien pour apprendre à réaliser des filtres dans Pandas:
##https://bitbucket.org/hrojas/learn-pandas/src/master/
##Lesson 3

In [ ]:
#Corriger la ou les données incohérentes
# 1) on corrige en mettant les valeurs de stock_status 2 dans stock_status
df_erp["stock_status"] = df_erp["stock_status_2"]

# 2) on vérifie qu'il n'y ai plus de différences
somme_stock_status = df_erp["stock_status"] == df_erp["stock_status_2"]
print("Nombre de lignes identiques : ", somme_stock_status.sum(), "sur 825")

# 3) On vérifie s'il y a des valeurs négatives
mask = df_erp["stock_quantity"] < 0
print("\n", "Lignes avec des valeurs négatives : ")
df_erp[mask]





## 2.1.1 - Analyse exploratoire de chaque variable du fichier erp.xlsx

### 2.1.1.1 - Analyse de la variable PRIX

In [ ]:
###############
## LES PRIX  ##
###############

#Vérification des prix: Y a t-il des prix non renseignés, négatifs ou nuls?
#Afficher le ou les prix non renseignés dans la colonne "price"
print("Nombres d'articles avec un prix non renseigné: {}".format(df_erp['price'].isnull().sum())) # isnull renvoie booléen vrai faux puis somme donc si 0 pas de valeur non renseignées
#Afficher le prix minimum de la colonne "price"
print("Prix minimal : {}".format(df_erp['price'].min()),"€")
#Afficher le prix maximum de la colonne "price"
print("Prix maximal : {}".format(df_erp['price'].max()),"€")
#Afficher les prix inférieurs à 0 (qu'est-ce qu'il faut en faire ?) il s'agit probablement d'une erreur donc on peut les supprimer
mask = df_erp['price'] < 0
display(df_erp[mask])
# On corrige en supprimant les valeurs négatives
df_erp = df_erp[~mask]  # ~veut dire que l'on prends toutes les valeurs non présentent dans le mask
df_erp



### 2.1.1.2 - Analyse de la variable STOCK

In [ ]:
#######################
### stock_quantity  ###
#######################

#Vérification de la colonne stock quantity
#Afficher la quantité minimum de la colonne "stock_quantity"
print("Stock minimal : {}".format(df_erp['stock_quantity'].min()))

#Afficher la quantité maximum de la colonne "stock_quantity"
print("Stock maximal : {}".format(df_erp['stock_quantity'].max()))

#Afficher les stocks inférieurs à 0 (qu'est-ce qu'il faut en faire ?) un stock ne peut etre négatif donc on le met égale à 0 car comme vu plus haut ces stocks étaient notés outofstock
mask = df_erp["stock_quantity"] < 0
display(df_erp[mask])

#Remplacer les valeurs de stock négatives par 0
df_erp.loc[mask, 'stock_quantity'] = 0

### 2.1.1.3 - Analyse de la variable ONSALE_WEB

In [ ]:
#Vérification de la colonne onsale_web et des valeurs qu'elle contient. Que signifient-elles? si 1 en vente sur le site sinon si 0 pas en vente sur le site
df_erp["onsale_web"].unique()

#Calculer le nombre d'articles en vente sur le site
print("Nombre d'articles en vente sur le site : {}".format(df_erp["onsale_web"].sum()))


In [ ]:
#Quelles sont les colonnes à conserver selon vous? Toutes les colonnes sauf celle que l'on a crée et onsale_web


In [ ]:
#Supprimer la colonne comportant le libellé "stock_status_2" car elle est redondante
#avec la colonne "stock_status".
df_erp = df_erp.drop(columns=["stock_status_2","onsale_web"])
df_erp


### 2.1.1.4 - Analyse de la variable prix d'achat

In [ ]:
######################
##   prix d'achat   ##
######################

#Vérification de la colonne purchase_price :
#Afficher le ou les prix non renseignés dans la colonne "purchase_price"
print("Nombres d'articles avec un prix d'achat non renseigné: {}".format(df_erp['purchase_price'].isnull().sum()))

#Afficher le prix minimum de la colonne "purchase_price"
print("Prix d'achat minimal : {}".format(df_erp['purchase_price'].min()),"€")

#Afficher le prix maximum de la colonne "purchase_price"
print("Prix d'achat maximal : {}".format(df_erp['purchase_price'].max()),"€")

## 2.2 - Analyse exploratoire du fichier web.xlsx


In [ ]:
#Dimension du dataset
print("Le tableau comporte {} observation(s) ou article(s)".format(df_web.shape[0]))
print("Le tableau comporte {} colonne(s)".format(df_web.shape[1]))

#Nombre d'observations
print("Nb observations : ",df_web.shape[0])

#Nombre de caractéristiques
print("Nb caractéristiques : ",df_web.shape[1])

In [ ]:
#Consulter le nombre de colonnes
print(df_web.shape[1])
#La nature des données dans chacune des colonnes
print(df_web.dtypes)
#Le nombre de valeurs présentes dans chacune des colonnes
print(df_web.count())


In [ ]:
#Selon vous, quelles sont les colonnes à conserver ? Toutes les colonnes ou il y a des infos non-vides et en lien avec les produits mais on supprime ce qui est en lien avec les posts (RGPD)


In [ ]:
#Si vous avez défini des colonnes à supprimer, effectuer l'opération
df_web = df_web.drop(columns=["tax_class", "virtual", "ping_status", "tax_status", "post_name", "post_modified", "post_modified_gmt", "post_parent", "guid", "menu_order", "post_mime_type", "comment_count", "downloadable", "rating_count", "average_rating", "post_author", "post_date", "post_date_gmt", "post_excerpt", "post_status", "comment_status","post_content","post_password","post_content_filtered"])



In [ ]:
#Visualisation des valeurs de la colonne sku
df_web["sku"].unique()

#Quelles sont les valeurs qui ne semblent pas respecter la régle de codification? Les valeurs vides et non numériques



In [ ]:
#Si vous avez identifié des codes articles ne respectant pas la régle de codification, consultez-les

# 1) Les lignes non numériques
mask_non_numeric = pd.to_numeric(df_web['sku'], errors='coerce').isna() & df_web['sku'].notna() #affiche les valeurs non-vides d'origine et qui sont vides après to_numeric car elles n'étaient pas numériques
display(df_web.loc[mask_non_numeric].head())

# 2)Couper la partie après le '-'
# extract before first dash, strip espaces et normalise les blancs invisibles
clean = (df_web['sku'].astype(str).str.partition('-')[0])   # récupère tout avant le premier '-'

# convertir en numérique puis en Int64 nullable (garde <NA> pour non-convertibles)
df_web['sku'] = pd.to_numeric(clean, errors='coerce').astype('Int64') #les valeurs textuelles von aussi devenir Nan

# vérif
print("\n", "Le type de la colonne sku est :", df_web['sku'].dtype, "\n")

# ça marque les valeurs qui existent mais ne sont pas numériques
display(df_web.loc[mask_non_numeric].head()) #on voit que la valeur avec le tiret est corrigé et le bon a été remplacé par nan donc c'est tout bon




In [ ]:
#Identifier les lignes sans code article
# on va devoir supprimer ses lignes car inexploitables si l'on a pas le sku (qui permet de faire la liaison entre les tables)

# Repérér combien et ou elles sont #il y en avait à l'origine 85 mais on en a 2 de plus du fait des bon cadeau que l'on a transformé en Nan
mask_to_drop = df_web["sku"].isna()
print("Nombre de lignes à supprimer :", mask_to_drop.sum())
display(df_web.loc[mask_to_drop].head(10))  # aperçu

In [ ]:
#Les lignes sans code article semblent être toutes non renseignées
#Pour s'en assurer, réaliser les étapes suivantes:
#1 - Créer un dataframe avec uniquement les lignes sans code article
mask_to_drop = df_web["sku"].isna()
df_web_vide = df_web.loc[mask_to_drop].reset_index(drop=True)

#2 - Utiliser la fonction df.info() sur ce nouveau dataframe pour observer le nombre de valeurs renseignées dans chacune des colonnes
df_web_vide.info()

#3 - Que constatez-vous? Sur les 87 non nulles  seulement 4 contiennent des infos (dont deux étaient des bons) et le total des ventes est négatif pour les 2 autres ce qui ne fait pas sens on peut donc les retirer
df_web_vide[df_web_vide['total_sales'].notna()]



In [ ]:
#Pour les codes articles identifiés, réaliser une analyse et définir l'action à entreprendre (on va supprimer les lignes vides)
#et pour l'autre on a enlevé le -1 qui est une erreur et on fera la moyenne de total_sales afin d'en avoir une seule car sku doit être unique

# 1) Supprimer les lignes vides
mask_to_drop = df_web["sku"].isna()
display(df_web.loc[mask_to_drop].head())
df_web = df_web.loc[~mask_to_drop].reset_index(drop=True) # ~ permet de prendre toutes les lignes qui ne sont pas dans le mask

# 2) Vérifier
display(df_web)





In [ ]:
#La clé pour chaque ligne est-elle unique? autrement dit, y a-t-il des doublons?
duplicats = df_web["sku"].value_counts()
duplicats = duplicats[duplicats > 1]
print(duplicats.head(20)) #seul le doublons 13127 est en 4 fois les autres toujours en 2 fois et a chaque fois pour l'un est une image l'autre est une ligne produit dans post_type

#1) On supprime toutes les lignes où sku est dupliqué ET post_type est différent de product
mask = (df_web["sku"].isin(duplicats.index))  & (df_web["post_type"] != "product") # duplicats.index = liste des sku qui sont en doublon, on prends donc les doublons qui n'ont pas product dans post_type
df_web = df_web.loc[~mask].reset_index(drop=True)

#2) On regarde a quoi ressemble nos 2 lignes avec les doublons restantes
print(df_web.loc[df_web['sku'].astype(str).str.contains('13127')])

#3) On fait la moyenne pour le dernier doublons afin de s'en débarasser
df_web = (df_web.groupby('sku', as_index=False)
              .agg({
                  'total_sales': 'mean',
                  'product_type': 'first',
                  'post_title': 'first',
                  'post_type' : 'first'
              })) # On fait ici une aggrégation sur sku en réalisant la moyenne de total_sales et en affichant la première valeur dispo pour les autres colonnes.
df_web['total_sales'] = df_web['total_sales'].round(1)

df_web = df_web.rename(columns={'sku': 'id_web'})
df_web = df_web.rename(columns={'post_title': 'nom_cuvee'})

print(df_web.dtypes) #on vérifie et on voit que désormais toutes les valeurs sont bien numériques
df_web


In [ ]:
#J'ai remarqué qu'il y avait de l'huile d'olive, je vais donc supprimer ces lignes car elle ne doivent pas faire partie d'une analyse sur le vin
mask = df_web['product_type'] == "Huile d'olive"

#combien de ligne correspondent au mask ?
print("Nombre de lignes à supprimer :", mask.sum())
display(df_web.loc[mask].head())  # aperçu
df_web = df_web.loc[~mask].reset_index(drop=True)
print(df_web['product_type'].unique())

#créer une liste avec toutes les id_web
liste_id_web = df_web['id_web'].tolist()
df_web


## 2.3 - Analyse exploratoire du fichier liaison.xlsx

In [ ]:
#Dimension du dataset
print("Le tableau comporte {} observation(s) ou article(s)".format(df_liaison.shape[0]))
print("Le tableau comporte {} colonne(s)".format(df_liaison.shape[1]))

#Nombre d'observations
print("Nb observations : ",df_liaison.shape[0])

#Nombre de caractéristiques
print("Nb caractéristiques : ",df_liaison.shape[1])


In [ ]:
#Consulter le nombre de colonnes
df_liaison.shape[1]
#La nature des données dans chacune des colonnes
df_liaison.dtypes
#Le nombre de valeurs présentes dans chacune des colonnes
df_liaison.count()


In [ ]:
#Les valeurs de la colonne "product_id" sont-elles toutes uniques?
df_liaison["product_id"].is_unique


In [ ]:
#Les valeurs de la colonne "id_web" sont-elles toutes uniques?
df_liaison["id_web"].is_unique

In [ ]:
#Avons-nous des articles sans correspondance? Oui

#combien de lignes vides pour id_web
print("Nombre de lignes vides pour id_web :", df_liaison["id_web"].isna().sum())
# 1) Supprimer les lignes vides
mask_to_drop = df_liaison["id_web"].isna()
df_liaison = df_liaison.loc[~mask_to_drop].reset_index(drop=True)
display(df_liaison)


# 2) Couper la partie après le '-'
clean = (df_liaison['id_web'].astype(str).str.partition('-')[0])   # récupère tout avant le premier '-'


# 3) Convertir en numérique puis en Integer
df_liaison['id_web'] = pd.to_numeric(clean, errors='coerce').astype('Int64') #il y a aussi une ligne avec bon cadeau qui sera supprimé mais cela ne correspond pas à un produit que l'on vends donc on peut le supprimer.

# 4) Afficher les doublons
df_liaison[df_liaison.duplicated(subset=['id_web'], keep=False)] # certains produits de l'ERP correspondent au même produit sur ID_web, a nettoyer après la jointure des tables
#keep = false affiche toutes les occurences des doublons



# Etape 3 - Jonction des fichiers

## Etape 3.1 - Jonction du fichier df_erp et df_liaison

In [ ]:
#Fusion des fichiers df_erp et df_liaison
df_merge = pd.merge(df_erp, df_liaison, on='product_id', how='left', indicator=True)

# statistiques rapides
print(df_merge['_merge'].value_counts())

# lignes de df_erp qui n'ont pas matché (left_only)
unmatched_left = df_merge[df_merge['_merge'] == 'left_only']
print("Lignes df_erp sans correspondant dans df_liaison :", len(unmatched_left))
display(unmatched_left.head())



In [ ]:
#Y a t-il des lignes ne "matchant" pas entre les 2 fichiers? oui il y en a 88


## Etape 3.2 - Jonction du fichier df_merge et df_web

In [ ]:
#Fusionner les datasets df_merge et df_web
df_merge_global = pd.merge(df_merge, df_web, on='id_web', how='left')
display(df_merge_global)
print(df_merge_global["id_web"].isna().sum())

# Afficher les lignes doublons pour id_web
display(df_merge_global[df_merge_global.duplicated(subset=['id_web'], keep=False)])

# Pour chacune des lignes doublons agréger par id_web et faire la moyenne pour price, stock_quantity et purchase_price pour le reste garder la première valeur supprime aussi les ligne vides, permet de nettoyer les doublons de id_web
df_merge_global = df_merge_global.groupby('id_web', as_index=False).agg({
              'product_id': 'first',
              'price': 'mean',
              'stock_quantity': 'mean',
              'stock_status': 'first',
              'purchase_price': 'mean',
              '_merge': 'first',
              'total_sales': 'first',
              'product_type': 'first',
              'nom_cuvee': 'first',
              'post_type' : 'first',})

print("lignes vides restantes : ", df_merge_global["id_web"].isna().sum())

#Les valeurs de la colonne "id_web" sont-elles toutes uniques?
print(df_merge_global["id_web"].is_unique) #oui on est tout bon



In [ ]:
#Avons-nous des lignes sans correspondance? Non, nous les avons supprimé

# 1) Afficher les lignes pour laquelle id_web n'est pas dans liste_id_web du dataframe df_web
display(df_merge_global[~df_merge_global["id_web"].isin(liste_id_web)])  #13849, 7033 et 11258	sont de l'huile d'olive que l'on a supprimé, le reste n'a juste pas de correspondance dans web.

# 2) A combien de lignes cela corresponds
print(df_merge_global[~df_merge_global["id_web"].isin(liste_id_web)].shape[0], "lignes n'ont pas de correspondances dans la table web")

# 3) Supprimer ces lignes du dataset car elles n'ont pas de correspondance
df_merge_global = df_merge_global[df_merge_global["id_web"].isin(liste_id_web)].reset_index(drop=True)
df_merge_global






# Etape 4 - Analyse univariée des prix

## Etape 4.1 - Exploration par la visualisation de données

In [ ]:
#Création d'une boîte à moustache de la répartition des prix grâce à Pandas
df_merge_global.boxplot(column='price', vert=False, figsize=(14, 4))
plt.show()

In [ ]:
#Autre méthode avec plotly express
fig = px.box(df_merge_global, x='price', points=False, title='Distribution des prix', orientation='h')
fig.show()

## Etape 4.2 - Exploration par l'utilisation de méthodes statistiques

### Etape 4.2.1 - Identification par le Z-index

In [ ]:
#Calculer la moyenne du prix
moyenne_prix = round(df_merge_global['price'].mean(),2)
print("Le prix moyen est de :", moyenne_prix,"€")

#Calculer l'écart-type du prix
ecart_type_prix = round(df_merge_global['price'].std(ddof=0),2) #ddof=0 pour représenter la population globale et non pas un échantillon
print("L'écart-type est de :", ecart_type_prix,"€")

#Calculer le Z-score
df_merge_global['prix_z'] = round((df_merge_global['price'] - moyenne_prix) / ecart_type_prix,1)

#Afficher les irrégularités
df_merge_global[df_merge_global['prix_z'] > 3]

# Le z-score mesure de combien d'écart-type une valeur s'éloigne de la moyenne


In [ ]:
#Quel est le seuil prix dont le z-score est supérieur à 3?
seuil = round(3*ecart_type_prix + moyenne_prix,2)
print("Le seuil de prix correspondant à un z-score de 3 est :", seuil, "€") #pour rappel la moustache supérieure était de 83.7€ comme moins sensible aux outliers

### Etape 4.2.2 - Identification par l'intervalle interquartile

In [ ]:
#Utilisation de la fonction "describe" de Pandas pour l'étude des mesures de dispersion
round(df_merge_global['price'].describe(),2)

In [ ]:
#Définir un seuil pour les articles "outliers" en prix
#faire calcul ecart interquartile
ecart_interquartile = round(df_merge_global['price'].quantile(0.75) - df_merge_global['price'].quantile(0.25),2)
print("L'écart interquartile est de :", ecart_interquartile,"€")
# Calcul moustache supérieure
seuil = round(df_merge_global['price'].quantile(0.75) + 1.5*ecart_interquartile,2)
print("Le seuil est de :", seuil, "€")


In [ ]:
#Définir le nombre d'articles et la proportion de l'ensemble du catalogue "outliers"
outliers = df_merge_global[df_merge_global['price'] > seuil]
print(f"le nombre d'outliers probables est de : {len(outliers)}")
outliers_proportion = round(len(outliers) / len(df_merge_global),2)*100
print(f"la proportion d'outliers probables est de {outliers_proportion}%")



In [ ]:
#Selon vous, ces outliers sont-ils justifiés ? Comment le démontrer si cela est possible ?
top10 = df_merge_global.sort_values('price', ascending=False).head(10)
top10
# une recherche rapide sur internet avec le nom de la bouteille et on voit que les prix sont en effet corrects


# Etape 5 - Analyse univariée du CA, des quantités vendues, des stocks et de la marge ainsi qu'une analyse multivariée

## Etape 5.1 - Analyse des ventes en CA

In [ ]:
##############################
# Calculer le CA du site web #
##############################

#Créer une colonne calculant le CA par article
df_merge_global["ca_par_article"] = round(df_merge_global["price"] * df_merge_global["total_sales"],2)
display(df_merge_global)

#Calculer la somme de la colonne "ca_par_article"
CA_site = df_merge_global["ca_par_article"].sum()
print(f"Le CA du site est de {CA_site}€")
#Ce résultat correspond au chiffre d'affaire du site web


In [ ]:
###############################
# Palmarès des articles en CA #
###############################

#Effectuer le tri dans l'ordre décroissant du CA du dataset df_merge
df_merge_global = df_merge_global.sort_values("ca_par_article", ascending=False)

#Réinitialiser l'index du dataset par un reset_index
df_merge_global = df_merge_global.reset_index(drop=True)

#Afficher les 20 premiers articles en CA
top_20 = df_merge_global.head(20)

#Graphique en barre des 20 premiers articles avec plotly express
fig = px.bar(top_20,
             x='nom_cuvee',
             y='ca_par_article',
             title=f"Top 20 articles",
             hover_data=['ca_par_article'],
             labels={'nom_cuvee':'Bouteille', 'ca_par_article':'CA bouteille (€)'},
             text='ca_par_article')
fig.update_layout(height=900, width=1500)
fig.show()

In [ ]:
#############################
# Calculer le 20 / 80 en CA #
#############################

#Créer une colonne calculant la part du CA de la ligne dans le dataset
df_merge_global["part_ca"] = round(df_merge_global["ca_par_article"] / CA_site * 100,2)
df_merge_global

#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_merge_global["cum_part_ca"] = df_merge_global["part_ca"].cumsum()
df_merge_global

#Grâce aux deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% du CA
ca_80 = df_merge_global[df_merge_global["cum_part_ca"] <= 80]
print(f"Le nombre d'articles représentant 80% du CA est de {len(ca_80)}")

#Afficher la proportion que représente ce groupe d'articles dans le catalogue entier du site web
print(f"La proportion que représente ce groupe d'articles dans le catalogue entier du site web est de {round(len(ca_80)/len(df_merge_global)*100,2)}%")
df_merge_global.head(20)


## Etape 5.2 - Analyse des ventes en quantité

In [ ]:
#####################################
# Palmarès des articles en quantité #
#####################################

#Effectuer le tri dans l'ordre décroissant de quantités vendues du dataset df_merge
df_merge_global = df_merge_global.sort_values("total_sales", ascending=False)
df_merge_global

#Réinitialiser l'index du dataset par un reset_index
df_merge_global = df_merge_global.reset_index(drop=True)
df_merge_global

#Afficher les 20 premiers articles en quantité
top_20_vente = df_merge_global.head(20)
top_20_vente

#Graphique en barre des 20 premiers articles avec plotly express
fig = px.bar(top_20_vente,
             x='nom_cuvee',
             y='total_sales',
             title=f"Meilleures ventes",
             hover_data=['total_sales'],
             labels={'nom_cuvee':'Bouteille', 'total_sales':'Nb de vente'},
             text='total_sales')
fig.update_layout(height=900, width=1500)
fig.show()


In [ ]:
#############################
# Calculer le 20 / 80 en Nb ventes #
#############################

#Créer une colonne calculant la part en quantité de la ligne dans le dataset
df_merge_global["part_vente"] = round(df_merge_global["total_sales"] / df_merge_global["total_sales"].sum() * 100,2)
df_merge_global

#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_merge_global["cum_part_vente"] = df_merge_global["part_vente"].cumsum()
display(df_merge_global.head(20))

#Grâce aux deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% des ventes en quantité
vente_80 = df_merge_global[df_merge_global["cum_part_vente"] <= 80]
print(f"Le nombre d'articles représentant 80% des ventes en quantité est de {len(vente_80)}")

#Afficher la proportion que représente ce groupe d'articles dans le catalogue entier du site web
print(f"La proportion que représente ce groupe d'articles dans le catalogue entier du site web est de {round(len(vente_80)/len(df_merge_global)*100,2)}%")


## Etape 5.3 - Analyse des stocks

In [ ]:
######################################
# Calculer le nombre de mois de stock #
######################################

#Import de numpy, numpy déjà importé

#Création de la colonne Rotation de stock
df_merge_global["rotation_stock"] = round(df_merge_global["stock_quantity"] / df_merge_global["total_sales"],2)
df_merge_global

#Remplacement des "inf" par 0
df_merge_global = df_merge_global.replace([np.inf, -np.inf], 0) # On remplace ici tout ce qui a une valeur infini (comme lorsque division par 0) par 0 afin d'éviter les bugs par la suite
df_merge_global

#Effectuer le tri dans l'ordre décroissant du nombre de mois de stock dans le dataset df_merge
df_merge_global = df_merge_global.sort_values("rotation_stock", ascending=False)
display(df_merge_global)

#Graphique en barre du flop 20 des produits qui ont le plus de mois de stock
top_20_rotation = df_merge_global.head(20)

fig = px.bar(top_20_rotation,
             x='nom_cuvee',
             y='rotation_stock',
             title=f"20 produits avec le plus de mois de stock",
             hover_data=['total_sales'],
             labels={'nom_cuvee':'Bouteille', 'rotation_stock':'Nb de mois en stock en fonction des ventes'},
             text='rotation_stock')
fig.update_layout(height=900, width=1500)
fig.show()


In [ ]:
####################################
# Valorisation des stocks en euros #
####################################

#Création de la colonne Valorisation des stocks en euros
df_merge_global["valorisation_stock_euros"] = round(df_merge_global["stock_quantity"] * df_merge_global["purchase_price"],2)
df_merge_global

#Calculer la somme de la colonne "Valorisation_stock_euros"
stock_euros = round(df_merge_global["valorisation_stock_euros"].sum(),2)
print(f"La valeur du stock est de {stock_euros}€")

In [ ]:
##############################################
# Valorisation du nombre de produits en stock #
##############################################

#Calculer la somme de la colonne stock quantity
stock_quantity = df_merge_global["stock_quantity"].sum()
print(f"Le nombre de produits en stock est de {stock_quantity}")

## Etape 5.4 - Analyse du taux de marge

In [ ]:
############################
# Analyse du taux de marge #
############################

#Création de la colonne Prix HT (La taxe sur les boissons alcoolisées est de 20%)
df_merge_global["prix_ht"] = round(df_merge_global["price"] * 0.8,2)
df_merge_global




#Création de la colonne Taux de marge
df_merge_global["taux_marge"] = round(((df_merge_global["prix_ht"] - df_merge_global["purchase_price"])/df_merge_global["purchase_price"]*100),2)
df_merge_global

#Afficher le prix minimum de la colonne "taux_marge"
print(f"Le taux de marge minimum est de {df_merge_global['taux_marge'].min()}%")


#Afficher le prix maximum de la colonne "taux_marge"
print(f"Le taux de marge maximum est de {df_merge_global['taux_marge'].max()}%")


In [ ]:
#Affichage de la ligne avec un taux de marge inférieur à 0
df_merge_global[df_merge_global["taux_marge"] < 0] #Négatif car on vends moins chère que ce que l'on achète, erreur probable


In [ ]:
#Création d'un dataframe avec les taux positifs
df_marge_positive = df_merge_global[df_merge_global["taux_marge"] > 0]
df_marge_positive

#Afficher le prix minimum de la colonne "taux_marge"
print(f"Le taux de marge minimum est de {df_marge_positive['taux_marge'].min()}%")

#Afficher le prix maximum de la colonne "taux_marge"
print(f"Le taux de marge maximum est de {df_marge_positive['taux_marge'].max()}%")


In [ ]:
#Création d'un dataframe avec le taux de marge moyen par type de produit
df_marge_type = round(df_marge_positive.groupby("product_type", as_index=False)["taux_marge"].mean(),2)
df_marge_type = df_marge_type.sort_values("taux_marge", ascending=False)


#Affichage dans un graphique du taux de marge par type de produit

fig = px.bar(df_marge_type,
             x='product_type',
             y='taux_marge',
             title=f"Taux de marge par type de produit",
             hover_data=['taux_marge'],
             labels={'product_type':'Type de produit', 'taux_marge':'Marge (%)'},
             text_auto=True )

fig.show()


## Etape 5.5 - Analyse des corrélations entre les variables stock, sales et price

In [ ]:
############################
# Analyse des corrélations #
############################

#Importation de Seaborn
import seaborn as sns # C'est une librairie python basée sur matplotlib pour rendre les graphiques plus esthétiques et aussi ajouter des possibilités de graph mais reste statique alors que plotly express est interactif

#Création d'une heatmap de corrélation avec les variables stock, sales et price
#On peut également créer un mask pour n'afficher qu'une demi heatmap

# Choisir les colonnes
candidates = ['stock_quantity', 'total_sales', 'price']

# Création du coefficient de corrélation
correlation = df_merge_global[candidates].corr(method='spearman') # .corr() calcul la matrice de corrélation on utilise spearman ici et non pearson car on a beaucoup d'outliers et pearson y est sensible

# Mask pour ne garder que la moitié supérieure
mask2 = np.triu(np.ones_like(correlation, dtype=bool)) # np.ones_like va créer un array booléen a partir de correlation puis np.triu va sélectionner comme true la partie supérieur du triangle donc les données de la moitié supérieure
print(mask2)

# Création de la mheat map
plt.figure(figsize=(7, 5))
sns.heatmap(
    correlation,
    mask = mask2,
    annot=True,   #annot permet d'afficher les scores exacts sur le graph
)
plt.title("Matrice de corrélation — stock / sales / price")
plt.show()

#pourrait etre intéressant d'analyser la distribution pour savoir si utiliser pearson ou spearman

In [ ]:
#Que peut-on conclure des corrélations ? Attention corrélation différent de causalité !

# Stock / Ventes
# corrélation positive forte : en général, plus il y a de stock, plus les ventes sont élevées. C’est logique : les produits avec plus d’offre ont aussi plus de ventes

# Ventes / Prix
# corrélation négative forte : les bouteilles les plus chères se vendent moins (ou inversement, les plus vendues sont souvent moins chères). Ça ressemble à de la sensibilité au prix

# Stock / Prix
# corrélation négative forte : Moins de stock pour les bouteilles les plus chères. Logique car moins de vente également.

In [ ]:
print("Le kurtosis des prix est de :", round(df_merge_global['price'].kurtosis(),2)) #cette valeur montre que la distribution est plus pointue au centre et qu'il y a beaucoups d'outliers

print("Le kurtosis des ventes est de :", round(df_merge_global['total_sales'].kurtosis(),2)) #cette valeur montre que la distribution est proche d'une distribution normale

print("Le kurtosis du stock est de :", round(df_merge_global['stock_quantity'].kurtosis(),2)) #cette valeur montre que la distribution est plus pointue au centre et qu'il y a beaucoups d'outliers

## Etape 5.6 - Mise à disposition de la nouvelle table sur un fichier Excel

In [ ]:
#Mettre le dataset df_merge sur un fichier Excel
df_merge_global.to_excel('df_merge_global.xlsx', index=False)
#Cette étape peut être utile pour partager le résultat du dataset obtenu avec les équipes.
